# Official baseline: RobustRAG (certifiably robust RAG), run as published

**Why a separate notebook.** RobustRAG is not a context filter. It answers from each retrieved passage in isolation and
aggregates the answers (keyword aggregation or secure decoding). It needs its own pipeline, data format and
dependency pins (`transformers==4.40.1`), so it **cannot be a policy inside the RAGLedger harness without being rewritten**, and a rewrite is
not "the published method". This notebook runs the authors' code unchanged, on **their** benchmark (`open_nq`, first 100 queries, their `Poison` attack).

**What it gives the paper**

1. A reproduction check: the numbers you get should match the RobustRAG paper's `open_nq` rows for the same model. If they do not, your environment differs and you must say so.
2. A reference row. It is **not paired** with the RAGLedger matrix (different questions, a static single-shot threat model with no
   source history), so never put it in the same significance test. Report it in a separate table with its own caption.

**Requirements:** a GPU that holds a 7B model in fp16 (L4/A100). Model weights come from Hugging Face; Mistral/Llama may need a token (Colab secret `HF_TOKEN`).
Nothing here is verified by the project authors until you run it: the commands are copied from the authors' `run.sh`.

In [ ]:
import json, os, pathlib, subprocess, sys, shutil, time
assert subprocess.run(["nvidia-smi", "-L"], capture_output=True).returncode == 0, "GPU runtime required"
from google.colab import drive, userdata
drive.mount("/content/drive")
DRIVE = pathlib.Path("/content/drive/MyDrive/RAGLedger_real/baselines/robustrag"); DRIVE.mkdir(parents=True, exist_ok=True)
try: os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception: print("no HF_TOKEN secret set; open models only")

In [ ]:
SHA = "9bc35b2fa5fa7d1088383b2789ec19a512d316b1"
R = pathlib.Path("/content/RobustRAG")
if not (R / ".git").exists():
    subprocess.run(["git", "clone", "--quiet", "https://github.com/inspire-group/RobustRAG.git", str(R)], check=True)
subprocess.run(["git", "-C", str(R), "checkout", "--quiet", SHA], check=True)
assert subprocess.check_output(["git", "-C", str(R), "rev-parse", "HEAD"], text=True).strip() == SHA
# The authors tested torch 2.2.1 / transformers 4.40.1; requirements.txt pins only transformers. Install exactly what it asks for.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(R / "requirements.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nltk", "spacy"], check=True)
print("RobustRAG at", SHA)

## Commands (verbatim from the authors' `run.sh`, `open_nq`, 10 retrieved passages, corruption size 1)

`MODEL` is `mistral7b` or `llama7b`. The vanilla (undefended) run is included so you can compare defended and undefended under the same attack.

In [ ]:
MODEL = "mistral7b"
RUNS = {
  "no_attack_vanilla": ["--defense_method", "none", "--corruption_size", "0"],
  "poison_vanilla":    ["--defense_method", "none", "--corruption_size", "1", "--attack_method", "Poison"],
  "poison_decoding":   ["--defense_method", "decoding", "--corruption_size", "1", "--eta", "0.0", "--attack_method", "Poison"],
  "poison_keyword":    ["--defense_method", "keyword", "--corruption_size", "1", "--alpha", "0.3", "--beta", "3", "--attack_method", "Poison"],
}
for name, extra in RUNS.items():
    log = DRIVE / f"{MODEL}_{name}.log"
    if log.exists() and "FINISHED" in log.read_text()[-200:]:
        print("skip finished:", name); continue
    cmd = [sys.executable, "main.py", "--model_name", MODEL, "--dataset_name", "open_nq", "--top_k", "10", "--save_response"] + extra
    print(" ".join(cmd)); t0 = time.time()
    with open(log, "w") as fh:
        rc = subprocess.run(cmd, cwd=R, stdout=fh, stderr=subprocess.STDOUT).returncode
        fh.write(f"\nFINISHED rc={rc} elapsed_s={round(time.time() - t0)}\n")
    print("  rc", rc, "elapsed s", round(time.time() - t0))

In [ ]:
# The authors' script prints its metrics to the log. Read them; do not retype numbers into the paper from memory.
for log in sorted(DRIVE.glob(f"{MODEL}_*.log")):
    print("=====", log.name); print("\n".join(log.read_text().splitlines()[-12:]))
# Anything the script saved (responses, results) lives under the repo; copy it next to the logs.
for sub in ("result", "results", "log", "logs"):
    if (R / sub).exists(): shutil.copytree(R / sub, DRIVE / sub, dirs_exist_ok=True)
(DRIVE / "manifest.json").write_text(json.dumps({"repo": "inspire-group/RobustRAG", "sha": SHA, "model": MODEL, "dataset": "open_nq (authors' subset)",
    "note": "run as published; not paired with the RAGLedger matrix"}, indent=2))

## Reading the result

Report: the authors' vanilla vs. RobustRAG accuracy and attack success under `Poison`, **for the model you ran**, next to the RobustRAG paper's published values.
In the paper's text state three things explicitly: (a) different benchmark and threat model from RAGLedger's evolving-knowledge-base stream,
(b) no source history exists in RobustRAG's setting, so RAGLedger's trust ledger has nothing to learn from, (c) that this table is a reference, not a head-to-head.